In [0]:
import dlt
from pyspark.sql.functions import *

@dlt.table(
    name="03_gold.bridge_metrics",
    comment = "10 minute avg temperature, max vibration and max tilt angle per bridge with window start/end"
)
def bridge_metrics():
    #apply 2 minute watermark  to bound late data for stateful ops
    temp = (
        dlt.read_stream("02_silver.bridge_temperature")
        .withWatermark("event_time", "2 minutes")
    )

    vib = (
        dlt.readStream("02_silver.bridge_vibration")
        .withWatermark("event_time", "2 minutes")
    )

    tilt = (
        dlt.readStream("02_silver.bridge_tilt")
        .withWatermark("event_time", "2 minutes")
    )

    #Compute 10 minute tumbling average temperature, retaining metadata

    temp_agg = (
        temp
        .groupBy(
            window(col("event_time"),"10 minutes"),
            col("bridge_id"),
            col("name"),
            col("location")
        )
        .agg(
            avg(col("temperature")).alias("avg_temp")
        
        ).select(
            col("bridge_id"),
            col("name"),
            col("location"),
            col("window.start").alias("window_start"),
            col("window.end").alias("window_end"),
            round(col("avg_temp"),2).alias("avg_temp")
        )
    )
    #Compute 10 minute tumbling max vibration

    vib_agg = (
        vib
        .groupBy(window("event_time", "10 minutes"), col("bridge_id"))
        .agg(
            max(col("vibration")).alias("max_vibration")
        ).select(
            col("bridge_id"),
            col("window.start").alias("window_start"),
            col("window.end").alias("window_end"),
            col("max_vibration")
        )
    )

    #Compute 10 minute tumbling max tilt angle

    tilt_agg = (
        tilt
        .groupBy(window("event_time", "10 minutes"), col("bridge_id"))
        .agg(
            max(col("tilt_angle")).alias("max_tilt_angle")
            )
        .select(
            col("bridge_id"),
            col("window.start").alias("window_start"),
            col("window.end").alias("window_end"),
            col("max_tilt_angle")
        )

        )
    return(
        temp_agg.alias("t")
            .join(
                vib_agg.alias("v"),
                on = ["bridge_id","window_start","window_end"],
                how = "inner"
            )
            .join(
                tilt_agg.alias("ti"),
                on = ["bridge_id","window_start","window_end"],
                how = "inner"
            )
            .select(
                col("bridge_id"),
                col("name"),
                col("location"),
                col("window_start"),
                col("window_end"),
                round(col("avg_temp"),2).alias("avg_temp"),
                col("max_vibration"),
                col("max_tilt_angle")
            )
        )
            
    
            

        



   
    